# Environment spaces and score

Il notebook mostra `observation_space`, `action_space`, score e reward degli ambienti custom Gym.


In [ ]:
from pathlib import Path
import sys

WORKING_DIR = Path.cwd().resolve()
if (WORKING_DIR / "beam_optimization").is_dir():
    REPO_ROOT = WORKING_DIR
else:
    PROJECT_ROOT = WORKING_DIR
    while PROJECT_ROOT.name != "beam_optimization" and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent
    REPO_ROOT = PROJECT_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from beam_optimization.config.adige import (
    BEAM_STATE_FEATURES,
    ERROR_SCORE,
    PARAMETERS,
    SCORE_REFERENCES,
    SCORE_WEIGHTS,
    action_bounds,
    observation_dim,
    observation_stage_labels,
)


## How the environments are structured

These are custom Gymnasium environments for optimizing an accelerator beam.
The agent observes selected beam stages, changes the machine parameters,
and gets a reward when the beam score improves.

The shared logic lives in `BaseBeamEnv`:
- `observation_space`: the visible beam features
- `action_space`: one delta for every configurable TraceWin parameter
- `score`: scalar beam quality; higher is better
- `reward = score(t+1) - score(t)`
- `reset` and `step`: return the score through the `info` dictionary
- `render`: includes a dedicated score/reward figure

`SurrogateEnv` and `TraceWinEnv` expose the same interface; only the simulator underneath changes.


In [ ]:
try:
    from IPython.display import Markdown, display
except ImportError:
    display = None
    Markdown = None


def fmt(value):
    return f"{float(value):.6g}"


def markdown_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        lines.append("| " + " | ".join(str(cell) for cell in row) + " |")
    return "\n".join(lines)


def show(markdown):
    if display is None:
        print(markdown)
    else:
        display(Markdown(markdown))


stage_labels = observation_stage_labels()
obs_shape = (observation_dim(),)
action_low, action_high = action_bounds()
action_shape = action_low.shape

summary_rows = [
    [name, ", ".join(stage_labels), obs_shape, "float32", action_shape, "float32", '`info["score"]`']
    for name in ("SurrogateEnv", "TraceWinEnv")
]

feature_rows = [[idx, f"`{name}`", "yes"] for idx, name in enumerate(BEAM_STATE_FEATURES)]

action_rows = []
for idx, parameter in enumerate(PARAMETERS):
    action_rows.append([
        idx,
        f"`{parameter.name}`",
        f"`{parameter.key}`",
        parameter.marker,
        fmt(parameter.default),
        fmt(action_low[idx]),
        fmt(action_high[idx]),
    ])

score_rows = [
    ["Transmission", "`npart_ratio`", "+", fmt(SCORE_WEIGHTS["npart_ratio"]), "clipped to [0, 1]"],
    ["Emittance", "`ex`, `ey`", "-", fmt(SCORE_WEIGHTS["emittance"]), f'{fmt(SCORE_REFERENCES["ex"])} each'],
    ["Offset", "`abs(x0)`, `abs(y0)`", "-", fmt(SCORE_WEIGHTS["offset"]), f'{fmt(SCORE_REFERENCES["x0"])} each'],
    ["Angle", "`abs(x'0)`, `abs(y'0)`", "-", fmt(SCORE_WEIGHTS["angle"]), fmt(SCORE_REFERENCES["x'0"]) + " each"],
    ["Size", "`SizeX`, `SizeY`", "-", fmt(SCORE_WEIGHTS["size"]), f'{fmt(SCORE_REFERENCES["SizeX"])} each'],
]

score_formula = (
    "```text\n"
    "score = 100*clip(npart_ratio, 0, 1)\n"
    "        - 200*((ex - 0.05) + (ey - 0.05))\n"
    "        - (abs(x0) + abs(y0))\n"
    "        - (abs(x'0) + abs(y'0))\n"
    "        - ((SizeX - 5) + (SizeY - 5))\n"
    "reward(t) = score(t) - score(t-1)\n"
    "```"
)

report = "\n\n".join([
    "## Environment interface",
    markdown_table(
        ["Environment", "Observed stages", "Obs shape", "Obs dtype", "Action shape", "Action dtype", "Score output"],
        summary_rows,
    ),
    "## Features inside one observed beam stage",
    markdown_table(["Index", "Feature", "Used by score"], feature_rows),
    "## Score and reward",
    "The score is computed from the final beam stage. It is not part of `observation_space`; it is returned as `info[\"score\"]`. Higher is better.",
    score_formula,
    markdown_table(["Component", "Features", "Sign", "Weight", "Reference"], score_rows),
    f"A reference beam with full transmission scores **100**. Failed simulations receive **{fmt(ERROR_SCORE)}**.",
    f"## Action space: {len(PARAMETERS)} deltas on TraceWin parameters",
    f"Each action contains {len(PARAMETERS)} simultaneous parameter shifts.",
    markdown_table(
        ["Index", "Parameter", "TraceWin key", "Marker", "Default", "Action min", "Action max"],
        action_rows,
    ),
    "Note: `SurrogateEnv` and `TraceWinEnv` share these spaces and the same score function.",
])

show(report)
